<a href="https://colab.research.google.com/github/quarcs-lab/gdo4bolivia/blob/main/notebooks/esda_inequality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spatial inequality of poverty and night-time lights

Part of the [GeoDevelopment Observatory of Bolivia](https://quarcs-lab.github.io/gdo4bolivia/).

How unequal are Bolivia's municipalities, how much of that inequality is regional, and did it change as poverty fell? This notebook measures inequality in two indicators with the [`geo4bolivia`](https://quarcs-lab.github.io/geo4bolivia/) package:

- **Multidimensional poverty** (`sdg1_2_mpi`, Multidimensional Poverty Index, 0–1; **higher = poorer**) of the *Atlas de los ODS en Bolivia 2026* (SDSN Bolivia), computed from the population censuses of 2012 and 2024, for all **343 municipalities and indigenous autonomies**, in **both periods**.
- **Night-time lights per capita** (`ln_t400NTLpc`, the Hodrick–Prescott trend of log lights per person), every year from 2012 to 2020. Lights and the WorldPop population behind them were computed on the 2015 municipal map, `map339`, and cover its **339 units**: the 4 units created since have no data, and the 6 municipalities that later lost territory to them (Colquechaca, Mizque, Porco, Tomave, San Ignacio, Santa Ana) are described by their 2015 territory. Every result for lights is computed on those 339 units, on the 2015 map.

Outline:

1. **Load** the data (`geo.load`)
2. **Distributions** and maps (`geo.distribution`, `geo.map`, `geo.aggregate`)
3. **Theil index**, split between and within the 9 departments (`geo.theil_decomposition`)
4. **Gini coefficient** and the Lorenz curve (`geo.inequality`)
5. **Spatial Gini**: how much inequality lies between neighbouring municipalities (`geo.inequality` with `spatial=True`)
6. **Summary**

Each analysis returns a **result object**. Displaying it shows its main figure and a reading of the numbers; its parts are `.fig`, `.df` (the table behind the figure), `.gt` (a formatted table) and `.interpret()`.

## 0. Setup

In Google Colab the next cell installs the package (a few seconds); it brings the libraries it uses (libpysal, inequality, mapclassify, plotly). Locally, `uv sync` has installed it already.

In [1]:
import importlib.util

missing = [p for p in ["geo4bolivia"] if importlib.util.find_spec(p) is None]
if missing:
    !pip install -q {" ".join(missing)}

In [2]:
import warnings
warnings.filterwarnings("ignore")    # library notices; the text explains the ones that matter

import pandas as pd

import geo4bolivia as geo

MPI = "sdg1_2_mpi"       # Multidimensional Poverty Index (higher = poorer), 2012 and 2024
NTL = "ln_t400NTLpc"     # log night-time lights per capita, HP trend, 2012-2020

## 1. Load the data

Each municipality has one `geo_id` (e.g. `BO-04-17-04`, Nuestra Señora de La Paz). `geo.load()` returns a panel with one row per municipality and year; `groups=` adds the name of each municipality and of its department. Poverty is stored at the years 2012 and 2024, lights at every year from 2012 to 2020.

In [3]:
mpi = geo.load("sdg01", variables=MPI, groups=["department"])      # 343 units
ntl = geo.load("ntl", variables=NTL, groups=["department"])        # 339 units (map339)

for name, panel in [("poverty", mpi), ("lights", ntl)]:
    years = sorted(panel["year"].unique().tolist())
    print(f"{name}: {panel['geo_id'].nunique()} units, years {years}")
mpi.head()

poverty: 343 units, years [2012, 2024]
lights: 339 units, years [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]


,geo_id,name,year,department,sdg1_2_mpi
0,BO-01-01-01,Azurduy,2012,Chuquisaca,0.512
1,BO-01-01-01,Azurduy,2024,Chuquisaca,0.187
2,BO-01-01-02,Tarvita,2012,Chuquisaca,0.452
3,BO-01-01-02,Tarvita,2024,Chuquisaca,0.133
4,BO-01-02-01,Villa Serrano,2012,Chuquisaca,0.267


## 2. Distributions and maps

`geo.distribution()` overlays the histograms of the two periods and summarises each:

In [4]:
dist = geo.distribution(mpi, MPI)
dist

Between 2012 and 2024, the mean of **Multidimensional poverty index (0-1)** fell from 0.234 to 0.0633, and the median went from 0.227 to 0.053.

The spread across units narrowed: the standard deviation went from 0.106 to 0.0441, and the interquartile range (the middle half of the units) from 0.153 to 0.053.

**Multidimensional poverty index (0-1)** in 2024 averages 0.0633 across 343 units (median 0.053, standard deviation 0.0441). Half of the units lie between 0.03 and 0.083, and the full range runs from 0.003 to 0.296. The distribution is strongly right-skewed: a tail of high values pulls the mean above the median (skewness 1.44).

_These are associations, not causal effects. A causal reading needs a research design._

The table behind it, and the same summary for lights in their first and last year:

In [5]:
lights = geo.distribution(ntl, NTL, year=[2012, 2020])
pd.concat({"Multidimensional poverty": dist.df, "Log lights per capita": lights.df},
          names=["indicator"]).droplevel(1).round(3)

,year,n,mean,sd,min,p25,median,p75,max,skewness
indicator,,,,,,,,,,
Multidimensional poverty,2012,343,0.234,0.106,0.018,0.153,0.227,0.306,0.563,0.457
Multidimensional poverty,2024,343,0.063,0.044,0.003,0.030,0.053,0.083,0.296,1.441
Log lights per capita,2012,339,13.033,1.672,0.856,12.050,13.118,14.268,17.071,-1.659
Log lights per capita,2020,339,14.040,1.177,10.488,13.206,14.101,14.922,18.063,-0.261


Where are the poorest municipalities? One map per period, each in its own quintiles (`scheme="quantiles"`), so the darkest class is the poorest fifth of that period:

In [6]:
for year in [2012, 2024]:
    geo.map(mpi, MPI, year=year, scheme="quantiles", title=f"MPI quintiles, {year}").show()

The poorest fifth of each period, by department:

In [7]:
quintile = mpi.groupby("year")[MPI].transform(lambda values: pd.qcut(values, 5, labels=False))
poorest = mpi[quintile == 4]
pd.crosstab(poorest["department"], poorest["year"], margins=True, margins_name="Total")

year,2012,2024,Total
department,,,
Beni,3,5,8
Chuquisaca,13,10,23
Cochabamba,12,14,26
La Paz,9,9,18
Oruro,8,10,18
Pando,8,3,11
Potosí,15,17,32
Santa Cruz,1,0,1
Total,69,68,137


Poverty fell sharply: the municipal mean MPI dropped from 0.234 in 2012 (range 0.018 to 0.563) to 0.063 in 2024 (range 0.003 to 0.296). Its absolute spread narrowed too (standard deviation 0.106, then 0.044), but less than the mean, so relative to the mean it widened. In 2024 most municipalities bunch near zero with a long right tail (skewness 1.44, against 0.46 in 2012).

In both periods the poorest fifth lies mainly in Potosí, Chuquisaca, Cochabamba, La Paz and Oruro (57 of its 69 units in 2012, 60 of 68 in 2024). Pando had 8 units in the poorest fifth in 2012 and has 3 in 2024. Santa Cruz has one unit in it in 2012 and none in 2024, and Tarija none in either period.

Log lights per capita (339 units) average 13.0 in 2012, with the middle half between 12.1 and 14.3 and a few very dark units (the minimum is 0.86). By 2020 the mean is 14.0 and the standard deviation has fallen from 1.67 to 1.18.

Before decomposing, the department averages show how large the regional gaps are. The plain mean gives every municipality the same weight. `geo.aggregate()` computes the population-weighted mean: the metadata of `sdg01` say that the MPI aggregates with census population as weights, and the package loads them.

In [8]:
plain = mpi.pivot_table(index="department", columns="year", values=MPI).add_prefix("mean_mpi_")
weighted = geo.add_names(geo.aggregate(mpi, to="department", dataset="sdg01"))
weighted = weighted.pivot(index="name", columns="year", values=MPI).add_prefix("pop_weighted_mpi_")
dep_table = plain.join(weighted).join(mpi.groupby("department")["geo_id"].nunique().rename("units"))
dep_table.sort_values("mean_mpi_2024", ascending=False).round(3)

,mean_mpi_2012,mean_mpi_2024,pop_weighted_mpi_2012,pop_weighted_mpi_2024,units
department,,,,,
Chuquisaca,0.323,0.090,0.221,0.058,29
Potosí,0.279,0.083,0.224,0.063,42
Beni,0.250,0.075,0.229,0.072,20
Cochabamba,0.230,0.070,0.142,0.038,48
Oruro,0.235,0.069,0.129,0.037,35
Pando,0.315,0.068,0.236,0.049,15
La Paz,0.224,0.060,0.110,0.027,87
Santa Cruz,0.169,0.032,0.095,0.017,56
Tarija,0.118,0.027,0.079,0.018,11


In [9]:
country = geo.aggregate(mpi, to="country", dataset="sdg01")
print("Bolivia, population-weighted MPI:", dict(zip(country["year"], country[MPI].round(3))))
print("Bolivia, mean across municipalities:", mpi.groupby("year")[MPI].mean().round(3).to_dict())

Bolivia, population-weighted MPI: {2012: 0.133, 2024: 0.033}
Bolivia, mean across municipalities: {2012: 0.234, 2024: 0.063}


Chuquisaca has the highest mean MPI in both periods (0.323, then 0.090) and Tarija the lowest (0.118, then 0.027). Pando had the largest drop: from the second poorest department on average in 2012 (0.315) to the sixth in 2024 (0.068). In every department and in both periods the population-weighted MPI is below the plain mean: the populous municipalities, led by the capitals, are less poor than the small rural ones. For the country, the population-weighted MPI is 0.133 in 2012 and 0.033 in 2024, against means across municipalities of 0.234 and 0.063. The population-weighted figures approximate the Atlas's department values; they are not the Atlas's own figures.

Everything below treats each municipality as one observation, whatever its population: it measures inequality between municipalities, not between people.

## 3. Theil index

The Theil index is 0 under perfect equality and rises with inequality. Its key property is **additive decomposability**: with the municipalities grouped into departments, total inequality splits exactly into a *between-department* part (differences in department means) and a *within-department* part (differences among municipalities of the same department). Like the Gini below, it is a relative measure: multiplying every value by the same factor leaves it unchanged.

`geo.theil_decomposition()` computes the index and its two parts in each year. `by="department"` groups the municipalities by the department they lie in. With `permutations=`, it also tests the between part against random reassignments of the municipalities to groups of the same sizes.

In [10]:
theil = geo.theil_decomposition(mpi, MPI, by="department", permutations=999)
theil

The Theil index of **Multidimensional poverty index (0-1)** splits additively into inequality **between** the 9 department groups and inequality **within** them (between + within = total, exactly).

In the latest year (2024), the total Theil index is 0.219: about 21% of it lies between department groups and 79% within them — most inequality plays out among units inside the same department.

Over the window, the between-group share stayed at about 21%.

The permutation test on the between component (p = 0.001) indicates the observed grouping captures more inequality than random reassignments of units to groups typically do.

_These are associations, not causal effects. A causal reading needs a research design._

In [11]:
theil.df.round(4)

,year,theil,between,within,between_share,p_between
0,2012,0.1039,0.0222,0.0817,0.2132,0.001
1,2024,0.2192,0.0470,0.1722,0.2144,0.001


The same decomposition for lights, in every year from 2012 to 2020:

In [12]:
theil_ntl = geo.theil_decomposition(ntl, NTL, by="department", permutations=999)
theil_ntl.fig

In [13]:
theil_ntl.df.round(4)

,year,theil,between,within,between_share,p_between
0,2012,0.0094,0.0016,0.0079,0.1664,0.001
1,2013,0.0079,0.0014,0.0065,0.1775,0.001
2,2014,0.0067,0.0012,0.0055,0.1861,0.001
3,2015,0.0058,0.0011,0.0047,0.1924,0.001
4,2016,0.0050,0.0010,0.0040,0.1962,0.001
5,2017,0.0045,0.0009,0.0036,0.1968,0.001
6,2018,0.0040,0.0008,0.0033,0.1936,0.001
7,2019,0.0037,0.0007,0.0030,0.1863,0.001
8,2020,0.0035,0.0006,0.0029,0.1751,0.001


The Theil index of poverty **more than doubled**, from 0.104 in 2012 to 0.219 in 2024. Poverty fell everywhere, but proportionally less in the poorest municipalities, so the relative gaps between municipalities widened even as the absolute gaps narrowed.

**Most inequality is within departments**: 79% in both periods. Differences between department averages account for the other 21%, a share that random groups of the same sizes did not reach in 999 trials (pseudo p-value 0.001). The split is stable: inequality rose by about the same proportion between and within departments. Bolivia's municipal disparities are therefore mostly local: each department contains both poor and less poor municipalities.

For lights, 82% to 83% of the Theil index lies within departments in 2012 and in 2020; the between share is 17% in 2012, peaks at 20% in 2016 and 2017, and is 18% in 2020. The level of the index is another matter. It is far smaller than that of poverty (0.009 in 2012) and falls every year, to 0.004 in 2020, but neither fact means what it would for a variable in levels. Lights enter in logarithms, which compresses their dispersion by construction, and the Theil and Gini of a logged variable fall mechanically when lights grow everywhere by the same proportion, because the mean of the logs rises. The standard deviation of the logs does not have that problem, and it fell from 1.67 to 1.18 (section 2): dispersion in lights did narrow. Read the indices of lights for their split between and within departments, and for neighbours in section 5, not for their level.

## 4. Gini coefficient

The Gini coefficient (0 = equality, 1 = maximal inequality) is half the mean absolute difference between all pairs of municipalities, relative to the mean. `geo.inequality()` computes it in each year, with the Theil index and, on request, the coefficient of variation (`"cv"`: the standard deviation over the mean). `spatial=True` adds the spatial Gini of the next section, with 999 permutations.

In [14]:
ineq = geo.inequality(mpi, MPI, measures=("gini", "theil", "cv"), spatial=True, permutations=999)
ineq

Cross-sectional inequality in **Multidimensional poverty index (0-1)** is tracked across 343 units over 2 years (2012 to 2024).

The Gini index rose from 0.254 to 0.367 between the first and last year.

The Theil index rose from 0.104 to 0.219 between the first and last year.

The coefficient of variation rose from 0.45 to 0.696 between the first and last year.

With two years (2012 and 2024) the measures can be compared, but their trend cannot be tested: that needs three or more years.

In the latest year, inequality between *neighboring* units under the weights (queen contiguity (1 island(s) attached to nearest neighbor), row-standardized, n=343 (on map343)) contributes 0.00428 to the Gini — about 1% of the overall Gini; the rest comes from pairs that are not neighbors.

The permutation test (p = 0.001) indicates the non-neighbor component of inequality is larger than expected under spatial randomness — differences line up with geography rather than being scattered among neighbors.

On the Lorenz curve of 2024, the half of the units with the lowest values holds 25% of the total (32% in 2012) and the tenth with the highest values holds 25% (19%); with no inequality they would hold 50% and 10%.

_These are associations, not causal effects. A causal reading needs a research design._

`.gt` is the table of the measures. With two years it shows both and their change; a trend needs three or more years.

In [15]:
ineq.gt

GT(_tbl_data=                             Measure      2012      2024     Change Change (%)
0                         Gini index    0.2535     0.367    +0.1135      +44.8
1                        Theil index    0.1039    0.2192    +0.1153     +111.0
2           Coefficient of variation    0.4505    0.6964    +0.2459      +54.6
3  Spatial Gini (neighbor component)  0.002886  0.004282  +0.001396      +48.4, _body=<great_tables._gt_data.Body object at 0x1261c3a90>, _boxhead=Boxhead([ColInfo(var='Measure', type=<ColInfoTypeEnum.stub: 2>, column_label='Measure', column_align='left', column_width=None, stub_level=0), ColInfo(var='2012', type=<ColInfoTypeEnum.default: 1>, column_label='2012', column_align='right', column_width=None, stub_level=None), ColInfo(var='2024', type=<ColInfoTypeEnum.default: 1>, column_label='2024', column_align='right', column_width=None, stub_level=None), ColInfo(var='Change', type=<ColInfoTypeEnum.default: 1>, column_label='Change', column_align='right', column_width=None, stub_level=None), ColInfo(var='Change (%)', type=<ColInfoTypeEnum.default: 1>, column_label='Change (%)', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x1261c0a10>, _spanners=Spanners([]), _heading=Heading(title='Inequality: Multidimensional poverty index (0-1)', subtitle='2012 and 2024, 343 units', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1261c34d0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1261c2950>, _source_notes=['With two years the measures can be compared, but their trend cannot be tested: that needs three or more years.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1261c24d0>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_b

The **Lorenz curve** shows the same inequality as a picture. Municipalities are sorted from the lowest MPI to the highest; the curve gives the share of the total (the sum of the MPI over all municipalities) held by the lowest share of them. The further the curve lies below the diagonal, the higher the Gini.

In [16]:
ineq.fig_lorenz

In [17]:
for year in [2012, 2024]:
    low, top = ineq.lorenz_share(year, 0.5), 1 - ineq.lorenz_share(year, 0.9)
    print(f"{year}: the half of the municipalities with the lowest MPI holds {low:.0%} of the total;",
          f"the tenth with the highest holds {top:.0%}")

2012: the half of the municipalities with the lowest MPI holds 32% of the total; the tenth with the highest holds 19%
2024: the half of the municipalities with the lowest MPI holds 25% of the total; the tenth with the highest holds 25%


For lights the panel has nine years, so `geo.inequality()` also fits a trend to each measure. The Lorenz curve is left out: it describes shares of a total, and a sum of logarithms is not a total of anything.

In [18]:
ineq_ntl = geo.inequality(ntl, NTL, spatial=True, permutations=999, lorenz=False)
ineq_ntl

Cross-sectional inequality in **Log night-time lights per capita (trend)** is tracked across 339 units over 9 years (2012 to 2020).

The Gini index fell from 0.0675 to 0.047 between the first and last year.

The Theil index fell from 0.00942 to 0.00354 between the first and last year.

The log-trend of the Gini index is -0.0458 per year (statistically significant at the 1% level), so inequality by this measure is **narrowing**.

The log-trend of the Theil index is -0.124 per year (statistically significant at the 1% level), so inequality by this measure is **narrowing**.

In the latest year, inequality between *neighboring* units under the weights (queen contiguity (1 island(s) attached to nearest neighbor), row-standardized, n=339 (on map339)) contributes 0.000556 to the Gini — about 1% of the overall Gini; the rest comes from pairs that are not neighbors.

The permutation test (p = 0.001) indicates the non-neighbor component of inequality is larger than expected under spatial randomness — differences line up with geography rather than being scattered among neighbors.

_These are associations, not causal effects. A causal reading needs a research design._

In [19]:
ineq_ntl.df.round(4)

,year,n_units,gini,theil,gini_spatial,gini_spatial_p
0,2012,339,0.0675,0.0094,0.0008,0.001
1,2013,339,0.0638,0.0079,0.0007,0.001
2,2014,339,0.0604,0.0067,0.0007,0.001
3,2015,339,0.0573,0.0058,0.0007,0.001
4,2016,339,0.0546,0.0050,0.0006,0.001
5,2017,339,0.0521,0.0045,0.0006,0.001
6,2018,339,0.0501,0.0040,0.0006,0.001
7,2019,339,0.0484,0.0037,0.0006,0.001
8,2020,339,0.0470,0.0035,0.0006,0.001


The Gini of poverty rose from 0.254 in 2012 to 0.367 in 2024 (by 45%): the expected gap in MPI between two randomly drawn municipalities (twice the Gini) grew from about 51% to about 73% of the mean MPI. The coefficient of variation tells the same story (0.45, then 0.70). With two periods these are two levels to compare, not a trend that can be tested.

The Lorenz curve of 2024 lies below that of 2012. The half of the municipalities with the lowest MPI holds 32% of the total in 2012 and 25% in 2024; the tenth with the highest MPI holds 19%, then 25%. Remaining poverty is more concentrated in the poorest municipalities.

The Gini of log lights falls from 0.068 in 2012 to 0.047 in 2020, about 4.5% a year. The caveat of section 3 applies: part of that fall is the rising mean of the logs. The series is also a smoothed trend, so its years are not independent observations and the significance of the fitted trend overstates the evidence.

## 5. Spatial Gini

The spatial Gini (Rey and Smith, 2013) splits the Gini into the contribution of pairs of **neighbouring** municipalities and that of all other pairs. If inequality had no spatial structure, neighbours would differ as much as any random pair, and their share of the Gini would roughly equal their share of all pairs. A smaller share means neighbours are more alike than random pairs (positive spatial autocorrelation). The pseudo p-value compares the observed split with 999 random reshuffles of the values across the map.

`spatial=True` used the package's default weights, queen contiguity among the units with data: 343 for poverty (on the 2025 map) and 339 for lights (on the 2015 map they belong to), where a unit without data is nobody's neighbour. Only who is a neighbour matters here, not the size of the weights. The column `gini_spatial` is the part of the Gini owed to neighbouring pairs, and `gini_spatial_p` its pseudo p-value.

In [20]:
def neighbour_shares(result, w):
    """The neighbours' share of the Gini, next to their share of all pairs of units."""
    table = result.df[["year", "n_units", "gini", "gini_spatial", "gini_spatial_p"]].copy()
    table["neighbour share of Gini, %"] = 100 * table["gini_spatial"] / table["gini"]
    table["neighbour pairs, % of pairs"] = 100 * w.mean_neighbors / (w.n - 1)
    return table

print("Poverty:", ineq.w_spec)
print("Lights: ", ineq_ntl.w_spec)
shares = pd.concat({"Multidimensional poverty": neighbour_shares(ineq, geo.weights(mpi)),
                    "Log lights per capita": neighbour_shares(ineq_ntl, geo.weights(ntl))},
                   names=["indicator"])
shares[shares["year"].isin([2012, 2020, 2024])].droplevel(1).round(4)

Poverty: queen contiguity (1 island(s) attached to nearest neighbor), row-standardized, n=343 (on map343)
Lights:  queen contiguity (1 island(s) attached to nearest neighbor), row-standardized, n=339 (on map339)


,year,n_units,gini,gini_spatial,gini_spatial_p,"neighbour share of Gini, %","neighbour pairs, % of pairs"
indicator,,,,,,,
Multidimensional poverty,2012,343,0.2535,0.0029,0.001,1.1382,1.6129
Multidimensional poverty,2024,343,0.3670,0.0043,0.001,1.1666,1.6129
Log lights per capita,2012,339,0.0675,0.0008,0.001,1.1639,1.6338
Log lights per capita,2020,339,0.0470,0.0006,0.001,1.1833,1.6338


The other notebooks of this series use the 6 nearest neighbours instead of contiguity. `w=` takes any weights from `geo.weights()`:

In [21]:
w6 = geo.weights(mpi, method="knn", k=6)
neighbour_shares(geo.inequality(mpi, MPI, w=w6, permutations=999), w6).round(4)

,year,n_units,gini,gini_spatial,gini_spatial_p,"neighbour share of Gini, %","neighbour pairs, % of pairs"
0,2012,343,0.2535,0.0031,0.001,1.2209,1.7544
1,2024,343,0.3670,0.0047,0.001,1.2836,1.7544


With queen contiguity on the 343 units, pairs of neighbours are 1.61% of all pairs of municipalities, and under random reshuffling they would carry about the same share of the Gini. In fact they carry 1.14% of poverty inequality in 2012 and 1.17% in 2024. For lights (339 units) neighbours are 1.63% of pairs and carry 1.16% of the Gini in 2012 and 1.18% in 2020. The difference is significant in every case (pseudo p-value 0.001, the smallest that 999 permutations allow): **inequality is spatially structured**. Neighbours resemble each other, and almost all inequality (about 98.8% of the Gini) comes from pairs of municipalities that are not neighbours.

The 6 nearest neighbours give the same reading: they are 1.75% of all pairs and carry 1.22% of the Gini of poverty in 2012 and 1.28% in 2024. With both definitions the neighbour share rose slightly between the two periods, so neighbours became a little less alike relative to other pairs, in line with the lower Moran's I of 2024 in `esda`.

## 6. Summary

In [22]:
def summary(result, decomposition):
    table = result.df.merge(decomposition.df[["year", "between_share"]], on="year")
    table["neighbour_share"] = table["gini_spatial"] / table["gini"]
    cols = ["year", "n_units", "gini", "theil", "between_share", "neighbour_share", "gini_spatial_p"]
    return table[cols]

both = pd.concat({"Multidimensional poverty": summary(ineq, theil),
                  "Log lights per capita": summary(ineq_ntl, theil_ntl)}, names=["indicator"])
both[both["year"].isin([2012, 2020, 2024])].droplevel(1).round(4)

,year,n_units,gini,theil,between_share,neighbour_share,gini_spatial_p
indicator,,,,,,,
Multidimensional poverty,2012,343,0.2535,0.1039,0.2132,0.0114,0.001
Multidimensional poverty,2024,343,0.3670,0.2192,0.2144,0.0117,0.001
Log lights per capita,2012,339,0.0675,0.0094,0.1664,0.0116,0.001
Log lights per capita,2020,339,0.0470,0.0035,0.1751,0.0118,0.001


- Municipal **poverty fell sharply but became more unequal in relative terms**: the Gini rose from 0.25 to 0.37 and the Theil index from 0.10 to 0.22 between 2012 and 2024, because the poorest municipalities reduced their poverty proportionally less. The tenth of the municipalities with the highest MPI holds 25% of the total in 2024, against 19% in 2012.
- **Most inequality lies within departments**: 79% for poverty in both periods and 82% to 83% for lights. Departmental averages hide most of the disparities between municipalities.
- Inequality is **spatially structured** in both periods and for both indicators: neighbouring municipalities resemble each other, so the poor and the less poor municipalities form regional clusters that cut across department boundaries.
- Lights are measured in logs, so the level and the fall of their Gini and Theil indices are not comparable with those of poverty; their dispersion (the standard deviation of the logs) fell from 1.67 in 2012 to 1.18 in 2020.
- These are descriptions of how the indicators are distributed. They are associations, not evidence of what causes the gaps.

Function pages: [`distribution`](https://quarcs-lab.github.io/geo4bolivia/reference/distribution.html), [`map`](https://quarcs-lab.github.io/geo4bolivia/reference/map.html), [`aggregate`](https://quarcs-lab.github.io/geo4bolivia/reference/aggregate.html), [`theil_decomposition`](https://quarcs-lab.github.io/geo4bolivia/reference/theil_decomposition.html), [`inequality`](https://quarcs-lab.github.io/geo4bolivia/reference/inequality.html), [`weights`](https://quarcs-lab.github.io/geo4bolivia/reference/weights.html).

Related notebooks: `esda` and `esda_dependence` (Moran's I and LISA clusters in both periods) and `esda_heterogeneity` (GWR and MGWR of poverty on lights).